In [ ]:
# This cell installs all required libraries for the V100 environment.
# Using -q for a quieter installation output.
!pip install -q \
  "transformers @ git+https://github.com/huggingface/transformers" \
  "torch==2.3.1" \
  "torchvision==0.18.1" \
  "torchcodec==0.2.1" \
  "polars" \ 
  "datasets" \
  "accelerate" \
  "huggingface_hub" \
  "tensorboard" \
  "matplotlib" \
  "tqdm"

# Now, install the correct CUDA-enabled PyTorch version for the V100.
# We reinstall torch and torchvision to ensure the GPU version is used.
!pip install -q torch==2.3.1 torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cu121

print("✅ Dependencies installed successfully.")

In [ ]:
import torch
import torch.nn
import polars as pl
import tarfile
from pathlib import Path
from functools import partial
from typing import Tuple

# --- Primary Imports ---
from huggingface_hub import hf_hub_download, login
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchcodec.decoders import VideoDecoder
from torchcodec.samplers import clips_at_random_indices
from torchvision.transforms import v2
from transformers import VJEPA2ForVideoClassification, VJEPA2VideoProcessor
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# --- Device and Data Type Configuration for V100 ---
# This will automatically select "cuda" and use float16 for optimized performance.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"✅ Using device: {DEVICE}")
print(f"✅ Using data type: {DTYPE}")

In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "c04260d6-f8c5-4350-9178-ecb79e65ce61",
   "metadata": {},
   "outputs": [],
   "source": [
    "# This cell installs all required libraries for the V100 environment.\n",
    "# Using -q for a quieter installation output.\n",
    "!pip install -q \\\n",
    "  \"transformers @ git+https://github.com/huggingface/transformers\" \\\n",
    "  \"torch==2.3.1\" \\\n",
    "  \"torchvision==0.18.1\" \\\n",
    "  \"torchcodec==0.2.1\" \\\n",
    "  \"polars\" \\ \n",
    "  \"datasets\" \\\n",
    "  \"accelerate\" \\\n",
    "  \"huggingface_hub\" \\\n",
    "  \"tensorboard\" \\\n",
    "  \"matplotlib\" \\\n",
    "  \"tqdm\"\n",
    "\n",
    "# Now, install the correct CUDA-enabled PyTorch version for the V100.\n",
    "# We reinstall torch and torchvision to ensure the GPU version is used.\n",
    "!pip install -q torch==2.3.1 torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cu121\n",
    "\n",
    "print(\"\u2705 Dependencies installed successfully.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "b986cd04-f59c-41e2-9d64-b1e00086bedd",
   "metadata": {},
   "outputs": [],
   "source": [
    "import torch\n",
    "import torch.nn\n",
    "import polars as pl\n",
    "import tarfile\n",
    "from pathlib import Path\n",
    "from functools import partial\n",
    "from typing import Tuple\n",
    "\n",
    "# --- Primary Imports ---\n",
    "from huggingface_hub import hf_hub_download, login\n",
    "from torch.utils.data import Dataset, DataLoader\n",
    "from torch.utils.tensorboard import SummaryWriter\n",
    "from torchcodec.decoders import VideoDecoder\n",
    "from torchcodec.samplers import clips_at_random_indices\n",
    "from torchvision.transforms import v2\n",
    "from transformers import VJEPA2ForVideoClassification, VJEPA2VideoProcessor\n",
    "from tqdm.auto import tqdm\n",
    "import matplotlib.pyplot as plt\n",
    "\n",
    "# --- Device and Data Type Configuration for V100 ---\n",
    "# This will automatically select \"cuda\" and use float16 for optimized performance.\n",
    "DEVICE = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n",
    "DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32\n",
    "\n",
    "print(f\"\u2705 Using device: {DEVICE}\")\n",
    "print(f\"\u2705 Using data type: {DTYPE}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "4f57507d-ac9d-4a7d-a659-d83b1b2e390f",
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- Download and Extract Dataset ---\n",
    "print(\"Downloading and extracting dataset...\")\n",
    "try:\n",
    "fpath = hf_hub_download(\n",
    "    repo_id=\"sayakpaul/ucf101-subset\",\n",
    "    filename=\"UCF101_subset.tar.gz\",\n",
    "    repo_type=\"dataset\"\n",
    "except Exception as e:\n    print(f\"Error downloading dataset: {e}\")\n    raise\n",
    ")\n",
    "try:\n",
    "with tarfile.open(fpath) as t:\n",
    "    t.extractall(\".\")\n",
    "print(\"Dataset extracted successfully.\")\n",
    "except Exception as e:\n    print(f\"Error extracting dataset: {e}\")\n    raise\n",
    "\n",
    "# --- Use Polars to Manage Files ---\n",
    "dataset_root_path = Path(\"UCF101_subset\")\n",
    "all_video_files = [str(p) for p in dataset_root_path.glob(\"**/*.avi\")]\n",
    "\n",
    "df = pl.DataFrame({\"path\": all_video_files}).with_columns(\n",
    "    pl.col(\"path\").str.split(\"/\").list.get(1).alias(\"split\"),\n",
    "    pl.col(\"path\").str.split(\"/\").list.get(2).alias(\"label\")\n",
    ")\n",
    "\n",
    "# --- Create Label Mappings ---\n",
    "unique_labels = df[\"label\"].unique().sort().to_list()\n",
    "label2id = {label: i for i, label in enumerate(unique_labels)}\n",
    "id2label = {i: label for label, i in label2id.items()}\n",
    "df = df.with_columns(pl.col(\"label\").map_dict(label2id).alias(\"label_id\"))\n",
    "\n",
    "print(\"\\n--- Dataset Overview ---\")\n",
    "print(df.head())\n",
    "\n",
    "# --- Custom PyTorch Dataset Class ---\n",
    "class PolarsVideoDataset(Dataset):\n",
    "    def __init__(self, df: pl.DataFrame):\n",
    "        self.df = df\n",
    "\n",
    "    def __len__(self) -> int:\n",
    "        return len(self.df)\n",
    "\n",
    "    def __getitem__(self, idx: int) -> Tuple[VideoDecoder, int]:\n",
    "        row = self.df.row(idx, named=True)\n",
    "        decoder = VideoDecoder(row[\"path\"])\n",
    "        return decoder, row[\"label_id\"]\n",
    "\n",
    "# --- Create Datasets and DataLoaders ---\n",
    "train_df = df.filter(pl.col(\"split\") == \"train\")\n",
    "val_df = df.filter(pl.col(\"split\") == \"val\")\n",
    "test_df = df.filter(pl.col(\"split\") == \"test\")\n",
    "\n",
    "train_ds, val_ds, test_ds = PolarsVideoDataset(train_df), PolarsVideoDataset(val_df), PolarsVideoDataset(test_df)\n",
    "\n",
    "# --- Define Collate Function and Transforms ---\n",
    "def collate_fn(samples, frames_per_clip, transforms):\n",
    "    clips, labels = [], []\n",
    "    for decoder, lbl in samples:\n",
    "        clip = clips_at_random_indices(\n",
    "            decoder, num_clips=1, num_frames_per_clip=frames_per_clip, num_indices_between_frames=3\n",
    "        ).data\n",
    "        clips.append(clip)\n",
    "        labels.append(lbl)\n",
    "    videos = torch.cat(clips, dim=0)\n",
    "    videos = transforms(videos)\n",
    "    return videos, torch.tensor(labels)\n",
    "\n",
    "model_name = \"facebook/vjepa2-vitl-fpc16-256-ssv2\"\n",
    "processor = VJEPA2VideoProcessor.from_pretrained(model_name)\n",
    "CROP_SIZE = (processor.crop_size[\"height\"], processor.crop_size[\"width\"])\n",
    "FRAMES_PER_CLIP = 16\n",
    "\n",
    "train_transforms = v2.Compose([\n",
    "    v2.RandomResizedCrop(CROP_SIZE, antialias=True),\n",
    "    v2.RandomHorizontalFlip(),\n",
    "    v2.ToDtype(DTYPE, scale=True),\n",
    "    v2.Normalize(mean=processor.image_mean, std=processor.image_std),\n",
    "])\n",
    "eval_transforms = v2.Compose([\n",
    "    v2.Resize(CROP_SIZE, antialias=True),\n",
    "    v2.CenterCrop(CROP_SIZE),\n",
    "    v2.ToDtype(DTYPE, scale=True),\n",
    "    v2.Normalize(mean=processor.image_mean, std=processor.image_std),\n",
    "])\n",
    "\n",
    "batch_size = 8  # A V100 can handle a larger batch size\n",
    "num_workers = 2\n",
    "\n",
    "train_loader = DataLoader(\n",
    "    train_ds, batch_size=batch_size, shuffle=True,\n",
    "    collate_fn=partial(collate_fn, frames_per_clip=FRAMES_PER_CLIP, transforms=train_transforms),\n",
    "    num_workers=num_workers, pin_memory=True,\n",
    ")\n",
    "val_loader = DataLoader(\n",
    "    val_ds, batch_size=batch_size, shuffle=False,\n",
    "    collate_fn=partial(collate_fn, frames_per_clip=FRAMES_PER_CLIP, transforms=eval_transforms),\n",
    "    num_workers=num_workers, pin_memory=True,\n",
    ")\n",
    "print(\"\\n\u2705 DataLoaders created successfully.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "3a57b6ea-485a-4fca-bf9c-1df85fb599b2",
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- Hugging Face Login ---\n",
    "# You'll need a Hugging Face account and a User Access Token with \"write\" permissions.\n",
    "# Get one here: https://huggingface.co/settings/tokens\n",
    "login()\n",
    "\n",
    "# --- Model Initialization ---\n",
    "model = VJEPA2ForVideoClassification.from_pretrained(\n",
    "    model_name,\n",
    "    torch_dtype=DTYPE,\n",
    "    label2id=label2id,\n",
    "    id2label=id2label,\n",
    "    ignore_mismatched_sizes=True,\n",
    ").to(DEVICE)\n",
    "\n",
    "# --- Freeze Backbone & Set up Optimizer ---\n",
    "for param in model.vjepa2.parameters():\n",
    "    param.requires_grad = False\n",
    "\n",
    "optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)\n",
    "scaler = torch.cuda.amp.GradScaler() # For stable mixed-precision training\n",
    "\n",
    "# --- Evaluation Function ---\n",
    "def evaluate(loader, model, processor, device, dtype) -> float:\n",
    "    model.eval()\n",
    "    correct, total = 0, 0\n",
    "    with torch.inference_mode():\n",
    "        for vids, labels in tqdm(loader, desc=\"Evaluating\", leave=False):\n",
    "            inputs = processor(vids, return_tensors=\"pt\")\n",
    "            inputs = {k: v.to(device) for k, v in inputs.items()}\n",
    "            labels = labels.to(device)\n",
    "            with torch.autocast(device_type=device.split(\":\")[0] if \":\" in device else device, dtype=dtype):\n",
    "                logits = model(**inputs).logits\n",
    "            preds = logits.argmax(-1)\n",
    "            correct += (preds == labels).sum().item()\n",
    "            total += labels.size(0)\n",
    "    return correct / total\n",
    "\n",
    "# --- Training Loop ---\n",
    "num_epochs = 5\n",
    "writer = SummaryWriter(\"runs/vjepa2_finetune_ucf101\")\n",
    "print(\"\\n--- Starting Training ---\")\n",
    "\n",
    "for epoch in range(num_epochs):\n",
    "    model.train()\n",
    "    progress_bar = tqdm(train_loader, desc=f\"Epoch {epoch+1}/{num_epochs}\")\n",
    "    for vids, labels in progress_bar:\n",
    "        inputs = processor(vids, return_tensors=\"pt\")\n",
    "        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}\n",
    "        labels = labels.to(DEVICE)\n",
    "        with torch.autocast(device_type=DEVICE.split(\":\")[0] if \":\" in DEVICE else DEVICE, dtype=DTYPE):\n",
    "            try:\n",
    "                            outputs = model(**inputs, labels=labels)\n",
    "            loss = outputs.loss\n",
    "            except Exception as e:\n                print(f\"Error during forward pass: {e}\")\n                raise\n",
    "        scaler.scale(loss).backward()\n",
    "        scaler.step(optimizer)\n",
    "        scaler.update()\n",
    "        optimizer.zero_grad()\n",
    "        progress_bar.set_postfix({\"loss\": f\"{loss.item():.4f}\"})\n",
    "\n",
    "    # --- Validation after each epoch ---\n",
    "    val_acc = evaluate(val_loader, model, processor, DEVICE, DTYPE)\n",
    "    print(f\"Epoch {epoch+1} Validation Accuracy: {val_acc:.4f}\")\n",
    "    # Save model checkpoint\n    torch.save(model.state_dict(), f\"model_checkpoint_epoch_{epoch+1}.pth\")\n",
    "    writer.add_scalar(\"Validation Accuracy\", val_acc, epoch + 1)\n",
    "\n",
    "writer.close()\n",
    "print(\"\\n\u2705 Training complete.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "679e8972-f1e0-440d-a4cc-4698cb212bf9",
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- Create Test Loader ---\n",
    "test_loader = DataLoader(\n",
    "    test_ds, batch_size=batch_size, shuffle=False,\n",
    "    collate_fn=partial(collate_fn, frames_per_clip=FRAMES_PER_CLIP, transforms=eval_transforms),\n",
    "    num_workers=num_workers, pin_memory=True,\n",
    ")\n",
    "\n",
    "# --- Final Test Evaluation ---\n",
    "test_acc = evaluate(test_loader, model, processor, DEVICE, DTYPE)\n",
    "print(f\"\u2705 Final Test Accuracy: {test_acc:.4f}\")\n",
    "\n",
    "# --- Push to Hub ---\n",
    "hub_model_id = \"dzeng/vjepa2-vitl-ucf101-finetuned\"\n",
    "print(f\"Pushing model to {hub_model_id}...\")\n",
    "model.push_to_hub(hub_model_id)\n",
    "processor.push_to_hub(hub_model_id)\n",
    "print(f\"\ud83d\ude80 Model pushed successfully!\")\n",
    "\n",
    "# --- Inference with Fine-Tuned Model ---\n",
    "print(\"\\n--- Testing with a new video ---\")\n",
    "video_url = \"https://huggingface.co/datasets/merve/vlm_test_images/resolve/main/IMG_3830.mp4\"\n",
    "vr = VideoDecoder(video_url)\n",
    "indices = torch.linspace(0, len(vr) - 1, FRAMES_PER_CLIP).long()\n",
    "video_frames = vr.get_frames_at(indices=indices.numpy()).data\n",
    "inputs = processor([video_frames], return_tensors=\"pt\")\n",
    "inputs = {k: v.to(DEVICE) for k, v in inputs.items()}\n",
    "\n",
    "with torch.inference_mode():\n",
    "    logits = model(**inputs).logits\n",
    "    predicted_idx = logits.argmax(-1).item()\n",
    "\n",
    "# The closest label in UCF-101 for a concert video is \"BandMarching\"\n",
    "print(f\"Predicted class: {model.config.id2label[predicted_idx]}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "a2c1cc99-d803-4c1b-b8e3-759ab9e8a63a",
   "metadata": {},
   "outputs": [],
   "source": []
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "vjepa-local",
   "language": "python",
   "name": "vjepa-local"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.12.11"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
# Save the final trained model locally
torch.save(model.state_dict(), "final_trained_model.pth")
print("Final trained model saved locally.")

In [ ]:
# --- Hugging Face Login ---
# You'll need a Hugging Face account and a User Access Token with "write" permissions.
# Get one here: https://huggingface.co/settings/tokens
login()

# --- Model Initialization ---
model = VJEPA2ForVideoClassification.from_pretrained(
    model_name,
    torch_dtype=DTYPE,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
).to(DEVICE)

# --- Freeze Backbone & Set up Optimizer ---
for param in model.vjepa2.parameters():
    param.requires_grad = False

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)
scaler = torch.cuda.amp.GradScaler() # For stable mixed-precision training

# --- Evaluation Function ---
def evaluate(loader, model, processor, device, dtype) -> float:
    model.eval()
    correct, total = 0, 0
    with torch.inference_mode():
        for vids, labels in tqdm(loader, desc="Evaluating", leave=False):
            inputs = processor(vids, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            labels = labels.to(device)
            with torch.autocast(device_type=device.split(":")[0] if ":" in device else device, dtype=dtype):
                logits = model(**inputs).logits
            preds = logits.argmax(-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

# --- Training Loop ---
num_epochs = 5
writer = SummaryWriter("runs/vjepa2_finetune_ucf101")
print("\n--- Starting Training ---")

for epoch in range(num_epochs):
    model.train()
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for vids, labels in progress_bar:
        inputs = processor(vids, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        labels = labels.to(DEVICE)
        with torch.autocast(device_type=DEVICE.split(":")[0] if ":" in DEVICE else DEVICE, dtype=DTYPE):
            try:
                            outputs = model(**inputs, labels=labels)
            loss = outputs.loss
            except Exception as e:
                print(f"Error during forward pass: {e}")
                raise
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    # --- Validation after each epoch ---
    val_acc = evaluate(val_loader, model, processor, DEVICE, DTYPE)
    print(f"Epoch {epoch+1} Validation Accuracy: {val_acc:.4f}")
    # Save model checkpoint
    torch.save(model.state_dict(), f"model_checkpoint_epoch_{epoch+1}.pth")
    writer.add_scalar("Validation Accuracy", val_acc, epoch + 1)

writer.close()
print("\n✅ Training complete.")

In [ ]:
# --- Create Test Loader ---
test_loader = DataLoader(
    test_ds, batch_size=batch_size, shuffle=False,
    collate_fn=partial(collate_fn, frames_per_clip=FRAMES_PER_CLIP, transforms=eval_transforms),
    num_workers=num_workers, pin_memory=True,
)

# --- Final Test Evaluation ---
test_acc = evaluate(test_loader, model, processor, DEVICE, DTYPE)
print(f"✅ Final Test Accuracy: {test_acc:.4f}")

# --- Push to Hub ---
hub_model_id = "dzeng/vjepa2-vitl-ucf101-finetuned"
print(f"Pushing model to {hub_model_id}...")
model.push_to_hub(hub_model_id)
processor.push_to_hub(hub_model_id)
print(f"🚀 Model pushed successfully!")

# --- Inference with Fine-Tuned Model ---
print("\n--- Testing with a new video ---")
video_url = "https://huggingface.co/datasets/merve/vlm_test_images/resolve/main/IMG_3830.mp4"
vr = VideoDecoder(video_url)
indices = torch.linspace(0, len(vr) - 1, FRAMES_PER_CLIP).long()
video_frames = vr.get_frames_at(indices=indices.numpy()).data
inputs = processor([video_frames], return_tensors="pt")
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

with torch.inference_mode():
    logits = model(**inputs).logits
    predicted_idx = logits.argmax(-1).item()

# The closest label in UCF-101 for a concert video is "BandMarching"
print(f"Predicted class: {model.config.id2label[predicted_idx]}")